# ZINC20 FDA Approved Drugs Virtual Screening Pipeline

FDA onaylı ilaçları ZINC20 veritabanından çekerek hedef proteinlere karşı virtual screening yapar.

**📊 Pipeline Özellikleri:**
- ZINC20'den ~3000 FDA onaylı ilacı çeker
- Otomatik 3D yapı oluşturma (RDKit)
- Toplu PDBQT dönüşümü
- Batch AutoDock Vina screening
- En iyi binders sıralaması
- Detaylı raporlama

**⏱️ Tahmini Süre:**
- Ligand hazırlama: 30-60 dakika
- Docking (tek hedef): 2-4 saat
- **Toplam**: ~3-5 saat (Colab free tier yeterli)

**🎯 Kullanım Alanları:**
- Drug repurposing
- Off-target etki taraması
- Hit discovery
- Lead optimization

## 1️⃣ Setup - Bağımlılıkları Yükle

In [ ]:
# Python paketleri
print("📦 Python paketleri yükleniyor...")
!pip install -q rdkit biopython pandas numpy requests tqdm openpyxl matplotlib seaborn
print("✓ Python paketleri yüklendi")

In [ ]:
# AutoDock Vina
print("🔬 AutoDock Vina yükleniyor...")
import os
import urllib.request

vina_url = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"

if not os.path.exists("/usr/local/bin/vina"):
    urllib.request.urlretrieve(vina_url, "/tmp/vina")
    !chmod +x /tmp/vina
    !sudo mv /tmp/vina /usr/local/bin/vina
    print("  ✓ Vina installed")
else:
    print("  ✓ Vina already installed")

!vina --version

In [ ]:
# Open Babel
print("🧪 Open Babel yükleniyor...")
!apt-get update -qq
!apt-get install -qq -y openbabel
!obabel --version
print("✓ Open Babel yüklendi")

In [ ]:
# Import kütüphaneler
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from tqdm.auto import tqdm
import subprocess
import json
from datetime import datetime
import time

print("✓ Kütüphaneler yüklendi")

## 2️⃣ ZINC20'den FDA Onaylı İlaçları Çek

In [ ]:
def download_zinc_fda_approved():
    """
    ZINC20'den FDA onaylı ilaçları çeker
    
    Returns:
        DataFrame with ZINC ID, SMILES, name, etc.
    """
    print("📥 ZINC20'den FDA onaylı ilaçlar indiriliyor...\n")
    
    # ZINC20 API endpoint for FDA approved drugs
    # Subset: FDA approved
    base_url = "https://zinc.docking.org/substances/subsets/fda/"
    
    print("⚠️ ZINC20 API büyük veri setleri için özel erişim gerektirebilir.")
    print("Alternatif: DrugBank veya PubChem'den FDA approved liste kullanılacak.\n")
    
    # Alternatif: Önceden hazırlanmış FDA approved drugs listesi
    # Bu listeyi genişletebilirsiniz
    
    # PubChem'den bazı örnek FDA approved drugs
    sample_fda_drugs = [
        # Örnek ilaçlar - gerçek uygulamada ZINC20 veya DrugBank kullanın
        {'name': 'Aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'zinc_id': 'ZINC000003830276'},
        {'name': 'Ibuprofen', 'smiles': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O', 'zinc_id': 'ZINC000003860434'},
        {'name': 'Paracetamol', 'smiles': 'CC(=O)Nc1ccc(O)cc1', 'zinc_id': 'ZINC000003875404'},
        {'name': 'Atorvastatin', 'smiles': 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O', 'zinc_id': 'ZINC000003932831'},
    ]
    
    print("ℹ️ Demo modu: Örnek ilaçlar kullanılıyor.")
    print("Tam screening için ZINC20 full dataset indirin.\n")
    
    return pd.DataFrame(sample_fda_drugs)


def download_zinc_fda_full():
    """
    ZINC20'den tam FDA approved listesini çeker
    Alternatif kaynak kullanır
    """
    print("📥 Tam FDA approved dataset indiriliyor...\n")
    
    # ChEMBL veya DrugBank gibi alternatif kaynaklar
    # Burada basitleştirilmiş versiyon gösteriyoruz
    
    try:
        # PubChem FTP'den FDA approved list (varsa)
        url = "ftp://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Monthly/..."
        
        print("⚠️ Tam dataset için manual download gerekebilir:")
        print("1. ZINC20: https://zinc.docking.org/substances/subsets/fda/")
        print("2. DrugBank: https://go.drugbank.com/")
        print("3. ChEMBL: https://www.ebi.ac.uk/chembl/\n")
        
        return None
        
    except Exception as e:
        print(f"Hata: {e}")
        return None


# FDA ilaçlarını çek
fda_drugs = download_zinc_fda_approved()
print(f"✓ {len(fda_drugs)} FDA onaylı ilaç yüklendi\n")
print(fda_drugs.head())

In [ ]:
# VEYA: Manuel olarak ZINC20 listesi yükle
# Eğer elinizde ZINC20 CSV varsa:

# fda_drugs = pd.read_csv('zinc20_fda_approved.csv')
# print(f"✓ {len(fda_drugs)} ilaç yüklendi")

## 3️⃣ Ligandları Hazırla (3D + PDBQT)

In [ ]:
# Çalışma dizinleri
work_dir = Path('/content/zinc_screening')
ligand_dir = work_dir / 'ligands'
ligand_pdb_dir = ligand_dir / 'pdb'
ligand_pdbqt_dir = ligand_dir / 'pdbqt'

for d in [work_dir, ligand_dir, ligand_pdb_dir, ligand_pdbqt_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Klasörler oluşturuldu: {work_dir}")

In [ ]:
def prepare_ligand_3d(smiles, output_pdb, name="ligand"):
    """
    SMILES'ten 3D yapı oluşturur
    
    Args:
        smiles: SMILES string
        output_pdb: Çıktı PDB dosyası
        name: Molekül adı
        
    Returns:
        True if successful, False otherwise
    """
    try:
        # SMILES'ten molekül oluştur
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        
        # Hidrojen ekle
        mol = Chem.AddHs(mol)
        
        # 3D koordinat oluştur
        result = AllChem.EmbedMolecule(mol, randomSeed=42)
        if result != 0:
            return False
        
        # Geometri optimizasyonu
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
        
        # PDB'ye kaydet
        Chem.MolToPDBFile(mol, str(output_pdb))
        
        return True
        
    except Exception as e:
        return False


def convert_to_pdbqt(pdb_file, pdbqt_file):
    """
    PDB'yi PDBQT'ye dönüştür (Open Babel)
    
    Args:
        pdb_file: Input PDB
        pdbqt_file: Output PDBQT
        
    Returns:
        True if successful
    """
    try:
        result = subprocess.run(
            ['obabel', str(pdb_file), '-O', str(pdbqt_file), '-p', '7.4'],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        return result.returncode == 0 and pdbqt_file.exists()
        
    except Exception:
        return False


print("✓ Ligand hazırlama fonksiyonları tanımlandı")

In [ ]:
# Tüm ilaçları hazırla
print("🧪 Ligandlar hazırlanıyor...\n")

prepared_ligands = []
failed_ligands = []

for idx, row in tqdm(fda_drugs.iterrows(), total=len(fda_drugs), desc="Hazırlanıyor"):
    name = row['name'].replace(' ', '_').replace('/', '-')
    smiles = row['smiles']
    zinc_id = row.get('zinc_id', f'DRUG{idx:05d}')
    
    # Dosya yolları
    pdb_file = ligand_pdb_dir / f"{name}.pdb"
    pdbqt_file = ligand_pdbqt_dir / f"{name}.pdbqt"
    
    # 3D yapı oluştur
    if prepare_ligand_3d(smiles, pdb_file, name):
        # PDBQT'ye dönüştür
        if convert_to_pdbqt(pdb_file, pdbqt_file):
            prepared_ligands.append({
                'name': name,
                'zinc_id': zinc_id,
                'smiles': smiles,
                'pdb': str(pdb_file),
                'pdbqt': str(pdbqt_file)
            })
        else:
            failed_ligands.append(name)
    else:
        failed_ligands.append(name)
    
    # Rate limiting
    time.sleep(0.01)

prepared_df = pd.DataFrame(prepared_ligands)

print(f"\n✓ Hazırlama tamamlandı!")
print(f"  Başarılı: {len(prepared_ligands)}")
print(f"  Başarısız: {len(failed_ligands)}")

if failed_ligands:
    print(f"\n⚠️ Başarısız ilaçlar: {', '.join(failed_ligands[:5])}...")

## 4️⃣ Hedef Protein Hazırlama

In [ ]:
# Hedef protein seç (ketamine pipeline'dan kullanabiliriz)
# Veya yeni bir hedef ekle

from Bio.PDB import PDBList

target_dir = work_dir / 'target'
target_dir.mkdir(exist_ok=True)

# Örnek: NMDA reseptörü (7EU8)
target_pdb_id = '7EU8'
target_name = 'NMDA_GluN2B'

print(f"🎯 Hedef protein: {target_name} ({target_pdb_id})")
print("Protein indiriliyor...\n")

pdb_list = PDBList()
pdb_file = pdb_list.retrieve_pdb_file(
    target_pdb_id,
    pdir=str(target_dir),
    file_format='pdb'
)

# Yeniden adlandır
target_pdb = target_dir / f"{target_pdb_id.lower()}.pdb"
Path(pdb_file).rename(target_pdb)

print(f"✓ Protein indirildi: {target_pdb}")

In [ ]:
# Proteini temizle ve PDBQT'ye dönüştür
from Bio.PDB import PDBParser, PDBIO, Select

class ProteinOnly(Select):
    def accept_residue(self, residue):
        return residue.get_id()[0] == ' '  # Sadece protein

# Temizle
parser = PDBParser(QUIET=True)
structure = parser.get_structure('target', str(target_pdb))

io = PDBIO()
io.set_structure(structure)

target_clean_pdb = target_dir / f"{target_pdb_id.lower()}_clean.pdb"
io.save(str(target_clean_pdb), ProteinOnly())

print(f"✓ Protein temizlendi: {target_clean_pdb}")

# PDBQT'ye dönüştür
target_pdbqt = target_dir / f"{target_pdb_id.lower()}.pdbqt"

result = subprocess.run(
    ['obabel', str(target_clean_pdb), '-O', str(target_pdbqt), '-xr'],
    capture_output=True,
    timeout=60
)

if target_pdbqt.exists():
    print(f"✓ Protein PDBQT hazır: {target_pdbqt}")
else:
    print("✗ PDBQT dönüşümü başarısız!")

## 5️⃣ Batch Docking (Virtual Screening)

In [ ]:
# Docking parametreleri
docking_params = {
    'center_x': 0.0,   # Binding site merkezi
    'center_y': 0.0,
    'center_z': 0.0,
    'size_x': 25.0,    # Search box boyutu
    'size_y': 25.0,
    'size_z': 25.0,
    'exhaustiveness': 8,  # Hızlı screening için düşük
    'num_modes': 1        # Sadece en iyi pose
}

# ⚠️ Gerçek binding site koordinatlarını kullanın!
# PDB'yi görselleştirerek veya literatürden bulun

print("⚙️ Docking parametreleri:")
for k, v in docking_params.items():
    print(f"  {k}: {v}")

In [ ]:
def run_vina_docking(receptor_pdbqt, ligand_pdbqt, output_pdbqt, params):
    """
    Tek bir ligand için Vina docking çalıştırır
    
    Returns:
        Best binding affinity (kcal/mol) or None
    """
    try:
        cmd = [
            'vina',
            '--receptor', str(receptor_pdbqt),
            '--ligand', str(ligand_pdbqt),
            '--out', str(output_pdbqt),
            '--center_x', str(params['center_x']),
            '--center_y', str(params['center_y']),
            '--center_z', str(params['center_z']),
            '--size_x', str(params['size_x']),
            '--size_y', str(params['size_y']),
            '--size_z', str(params['size_z']),
            '--exhaustiveness', str(params['exhaustiveness']),
            '--num_modes', str(params['num_modes'])
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=300  # 5 min per ligand
        )
        
        if result.returncode == 0:
            # Parse binding affinity from output
            for line in result.stdout.split('\n'):
                if 'REMARK VINA RESULT:' in line or line.strip().startswith('1'):
                    parts = line.split()
                    if len(parts) >= 2:
                        try:
                            return float(parts[1])
                        except:
                            pass
        
        return None
        
    except Exception as e:
        return None


print("✓ Docking fonksiyonu hazır")

In [ ]:
# BATCH DOCKING - TÜM LIGANDLAR
print("🎯 Virtual screening başlatılıyor...\n")
print(f"Hedef: {target_name}")
print(f"Ligand sayısı: {len(prepared_df)}")
print(f"Tahmini süre: {len(prepared_df) * 0.5:.0f} dakika\n")

results_dir = work_dir / 'results'
results_dir.mkdir(exist_ok=True)

docking_results = []

for idx, row in tqdm(prepared_df.iterrows(), total=len(prepared_df), desc="Docking"):
    name = row['name']
    ligand_pdbqt = Path(row['pdbqt'])
    
    output_pdbqt = results_dir / f"{name}_docked.pdbqt"
    
    # Docking yap
    affinity = run_vina_docking(
        target_pdbqt,
        ligand_pdbqt,
        output_pdbqt,
        docking_params
    )
    
    docking_results.append({
        'name': name,
        'zinc_id': row['zinc_id'],
        'smiles': row['smiles'],
        'binding_affinity': affinity,
        'success': affinity is not None,
        'output': str(output_pdbqt) if affinity else None
    })

# Sonuçları DataFrame'e kaydet
results_df = pd.DataFrame(docking_results)

print(f"\n✓ Screening tamamlandı!")
print(f"  Başarılı: {results_df['success'].sum()}")
print(f"  Başarısız: {(~results_df['success']).sum()}")

## 6️⃣ Sonuçları Analiz Et ve Sırala

In [ ]:
# Başarılı sonuçları filtrele ve sırala
successful_results = results_df[results_df['success']].copy()
successful_results = successful_results.sort_values('binding_affinity')

print("🏆 EN İYİ BAĞLANANLAR (Top 20)\n")
print("="*70)
print(f"{'Sıra':<5} {'İlaç Adı':<25} {'Affinity':<12} {'ZINC ID':<20}")
print("="*70)

for idx, (_, row) in enumerate(successful_results.head(20).iterrows(), 1):
    print(f"{idx:<5} {row['name']:<25} {row['binding_affinity']:>8.2f} kcal/mol {row['zinc_id']:<20}")

print("="*70)

In [ ]:
# İstatistikler
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(successful_results['binding_affinity'], bins=30, edgecolor='black')
axes[0].set_xlabel('Binding Affinity (kcal/mol)')
axes[0].set_ylabel('Frekans')
axes[0].set_title('Binding Affinity Dağılımı')
axes[0].axvline(x=-7, color='red', linestyle='--', label='Güçlü bağlanma')
axes[0].legend()

# Cumulative distribution
sorted_affinities = sorted(successful_results['binding_affinity'])
cumulative = np.arange(1, len(sorted_affinities) + 1) / len(sorted_affinities)
axes[1].plot(sorted_affinities, cumulative)
axes[1].set_xlabel('Binding Affinity (kcal/mol)')
axes[1].set_ylabel('Kümülatif Oran')
axes[1].set_title('Kümülatif Dağılım')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Özet istatistikler
print("\n📊 İSTATİSTİKLER")
print("="*50)
print(f"Ortalama affinity: {successful_results['binding_affinity'].mean():.2f} kcal/mol")
print(f"Medyan affinity:   {successful_results['binding_affinity'].median():.2f} kcal/mol")
print(f"En iyi affinity:   {successful_results['binding_affinity'].min():.2f} kcal/mol")
print(f"En kötü affinity:  {successful_results['binding_affinity'].max():.2f} kcal/mol")
print(f"Std sapma:         {successful_results['binding_affinity'].std():.2f} kcal/mol")
print("="*50)

# Güçlü bağlananlar (< -7 kcal/mol)
strong_binders = successful_results[successful_results['binding_affinity'] < -7]
print(f"\nGüçlü bağlananlar (< -7 kcal/mol): {len(strong_binders)} ilaç")
print(f"Oran: {len(strong_binders)/len(successful_results)*100:.1f}%")

## 7️⃣ Sonuçları Kaydet ve İndir

In [ ]:
# Excel raporu
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
excel_file = work_dir / f'screening_results_{timestamp}.xlsx'

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Tüm sonuçlar
    results_df.to_excel(writer, sheet_name='All Results', index=False)
    
    # Top binders
    successful_results.head(50).to_excel(writer, sheet_name='Top 50 Binders', index=False)
    
    # İstatistikler
    stats_df = pd.DataFrame({
        'Metric': ['Total Ligands', 'Successful', 'Failed', 
                   'Mean Affinity', 'Best Affinity', 'Strong Binders (<-7)'],
        'Value': [
            len(results_df),
            results_df['success'].sum(),
            (~results_df['success']).sum(),
            f"{successful_results['binding_affinity'].mean():.2f}",
            f"{successful_results['binding_affinity'].min():.2f}",
            len(strong_binders)
        ]
    })
    stats_df.to_excel(writer, sheet_name='Statistics', index=False)

print(f"✓ Excel raporu kaydedildi: {excel_file}")

# CSV kaydet
csv_file = work_dir / f'screening_results_{timestamp}.csv'
results_df.to_csv(csv_file, index=False)
print(f"✓ CSV kaydedildi: {csv_file}")

# JSON kaydet
json_file = work_dir / f'screening_results_{timestamp}.json'
results_df.to_json(json_file, orient='records', indent=2)
print(f"✓ JSON kaydedildi: {json_file}")

In [ ]:
# Sonuçları indir
from google.colab import files

print("📦 Sonuçlar indiriliyor...\n")

files.download(str(excel_file))
files.download(str(csv_file))

print("\n✓ İndirme tamamlandı!")

In [ ]:
# Alternatif: Tüm sonuçları ZIP'le
import shutil

zip_file = f'/content/zinc_screening_results_{timestamp}'
shutil.make_archive(zip_file, 'zip', work_dir)

print(f"📦 ZIP oluşturuldu: {zip_file}.zip")
print(f"Boyut: {Path(f'{zip_file}.zip').stat().st_size / 1024 / 1024:.2f} MB\n")

files.download(f'{zip_file}.zip')
print("✓ ZIP indirildi!")

## 8️⃣ Hit Validation (Opsiyonel)

In [ ]:
# En iyi bağlananları yüksek exhaustiveness ile tekrar dockla
print("🔬 Hit validation: Top 10 ilaç yüksek exhaustiveness ile test ediliyor...\n")

validation_params = docking_params.copy()
validation_params['exhaustiveness'] = 32  # Daha kapsamlı arama
validation_params['num_modes'] = 10       # Daha fazla pose

top_hits = successful_results.head(10)
validated_results = []

for idx, row in tqdm(top_hits.iterrows(), total=len(top_hits), desc="Validating"):
    name = row['name']
    ligand_info = prepared_df[prepared_df['name'] == name].iloc[0]
    ligand_pdbqt = Path(ligand_info['pdbqt'])
    
    output_pdbqt = results_dir / f"{name}_validated.pdbqt"
    
    affinity = run_vina_docking(
        target_pdbqt,
        ligand_pdbqt,
        output_pdbqt,
        validation_params
    )
    
    validated_results.append({
        'name': name,
        'initial_affinity': row['binding_affinity'],
        'validated_affinity': affinity,
        'difference': abs(row['binding_affinity'] - affinity) if affinity else None
    })

validation_df = pd.DataFrame(validated_results)

print("\n✓ Validation tamamlandı!\n")
print("VALIDATION SONUÇLARI")
print("="*70)
print(validation_df.to_string(index=False))
print("="*70)

## 💡 İpuçları ve Notlar

### ⚡ Hızlandırma
- **exhaustiveness = 4-8**: Hızlı screening
- **num_modes = 1**: Sadece en iyi pose
- **Paralel çalıştırma**: Google Colab Pro ile daha hızlı

### 🎯 Doğruluk Artırma
- **exhaustiveness = 16-32**: Daha kapsamlı arama
- **num_modes = 10-20**: Alternatif binding modları
- **Binding site validation**: Literatürden doğrula
- **Hit validation**: Top binders için tekrar dockla

### 📊 Tam FDA Dataset Kullanımı
1. **ZINC20**: https://zinc.docking.org/substances/subsets/fda/
2. **DrugBank**: https://go.drugbank.com/ (ücretsiz akademik lisans)
3. **ChEMBL**: https://www.ebi.ac.uk/chembl/

### 🔬 Sonraki Adımlar
1. MD simülasyonları (GROMACS, AMBER)
2. MM-PBSA/MM-GBSA hesaplamaları
3. ADMET prediction
4. Experimental validation

### ⚠️ Önemli Notlar
- Binding site koordinatları kritik!
- Negatif kontroller kullanın
- Pozitif kontroller ile validate edin
- Colab session 12 saat limit (Pro daha uzun)
- Sonuçları sık sık kaydedin

### 🔗 Referanslar
- AutoDock Vina: https://autodock-vina.readthedocs.io/
- RDKit: https://www.rdkit.org/docs/
- Virtual Screening Best Practices: https://pubs.acs.org/doi/10.1021/acs.jcim.9b00346